# U-Net like CNN without skip-connections (TEST)

## Imports
### Libs

In [ ]:
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from clearml import Dataset, Task, TaskTypes
from torch import nn
from torch.utils.data import DataLoader

### Chromatica modules

In [ ]:
from chromatica.datasets.dataset import ImageDataset
from chromatica.nn.v1.cnn import CNN

### Check for CUDA or MPS

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS is used")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is used")
else:
    device = torch.device("cpu")
    print("CPU is used")

## Test task init

In [ ]:
task = Task.init(
    project_name="Chromatica",
    task_name="Test U-Net like without skip-connections",
    task_type=TaskTypes.testing,
)

## Load dataset

In [ ]:
path = Path(
    Dataset.get(dataset_project="Colorization", dataset_name="Food101").get_local_copy()
)

In [ ]:
dataset = ImageDataset(path / "test")

In [ ]:
loader = DataLoader(
    dataset,
    num_workers=4,
    persistent_workers=True,
    pin_memory=(device == torch.device("cuda")),
)

## Test
### Load model

In [ ]:
train_task = Task.get_task(
    project_name="Chromatica",
    task_name="Train U-Net like without skip-connections",
)
model_artifact = train_task.artifacts["model"].get_local_copy()

In [ ]:
model = CNN()
model.load_state_dict(torch.load(model_artifact))
model = model.to(device)

### Testing

In [ ]:
%%time

criterion = nn.MSELoss()

total_loss = 0.0
loss_per_class = defaultdict(float)
count_per_class = defaultdict(int)

model.eval()
with torch.no_grad():
    for x_, y_, label_ in loader:
        x = x_.to(device, non_blocking=True)
        y = y_.to(device, non_blocking=True)
        label = int(label_.item())

        pred = model(x)
        loss = criterion(pred, y)
        loss_value = loss.item()

        total_loss += loss_value
        loss_per_class[label] += loss_value
        count_per_class[label] += 1

avg_loss = total_loss / sum(count_per_class.values())
task.get_logger().report_single_value("Avg. Loss", avg_loss)

labels = sorted(loss_per_class.keys())
normalized_losses = [
    loss_per_class[label] / count_per_class[label]
    for label in labels
    if label in count_per_class
]

xlabels = [str(label) for label in labels]

task.get_logger().report_histogram(
    title="Loss per Class",
    series="Normalized Loss",
    values=normalized_losses,
    iteration=0,
    xlabels=xlabels,
    xaxis="Class",
    yaxis="Loss",
)

## Visualization

In [ ]:
imgs = {}
labels = []
with torch.no_grad():
    for x, y, label_ in loader:
        label = int(label_.item())
        labels.append(label)
        if label not in imgs:
            imgs[label] = x, y

In [ ]:
x, y = imgs[labels[0]]

y = y[0]
x = x[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# TODO: Here should be util function, like lab2image
# https://github.com/snailUlitka/chromatica/issues/14

# Something like this: axes[0].imshow(lab2image(x, y))
axes[0].axis("off")
axes[0].set_title("Original")

pred = model(x[None, :, :, :].to(device))
# Something like this: axes[1].imshow(lab2image(x, pred[0].cpu()))
axes[1].axis("off")
axes[1].set_title("Predict")

plt.tight_layout()
plt.show()

In [ ]:
x, y = imgs[labels[1]]

y = y[0]
x = x[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# TODO: Here should be util function, like lab2image
# https://github.com/snailUlitka/chromatica/issues/14

# Something like this: axes[0].imshow(lab2image(x, y))
axes[0].axis("off")
axes[0].set_title("Original")

pred = model(x[None, :, :, :].to(device))
# Something like this: axes[1].imshow(lab2image(x, pred[0].cpu()))
axes[1].axis("off")
axes[1].set_title("Predict")

plt.tight_layout()
plt.show()

In [ ]:
x, y = imgs[labels[2]]

y = y[0]
x = x[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# TODO: Here should be util function, like lab2image
# https://github.com/snailUlitka/chromatica/issues/14

# Something like this: axes[0].imshow(lab2image(x, y))
axes[0].axis("off")
axes[0].set_title("Original")

pred = model(x[None, :, :, :].to(device))
# Something like this: axes[1].imshow(lab2image(x, pred[0].cpu()))
axes[1].axis("off")
axes[1].set_title("Predict")

plt.tight_layout()
plt.show()

In [ ]:
task.mark_completed()